In [ ]:

from google.colab import drive
import os

# 1️⃣ Mount Google Drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/fingerprint_saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# 2️⃣ Set paths
zip_path = "/content/drive/MyDrive/f_dataset.zip"  # Your uploaded zip file
extract_path = "/content/f_dataset"  # Where it will be unzipped

# 3️⃣ Unzip if not already done
if not os.path.exists(extract_path):
    print("Unzipping dataset...")
    !unzip -q "{zip_path}" -d "{extract_path}"
    print("Unzipping done!")
else:
    print("Dataset already unzipped.")

# 4️⃣ Verify folder structure
print("Blood group folders in dataset:")
!ls "{extract_path}"

# 5️⃣ Set DATA_DIR for your pipeline
DATA_DIR = extract_path
print(f"\nDATA_DIR set to: {DATA_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already unzipped.
Blood group folders in dataset:
A+  A-	AB+  AB-  B+  B-  O+  O-

DATA_DIR set to: /content/f_dataset


In [ ]:
"""
Enhanced Multi-Model Ensemble Pipeline for Fingerprint Blood Group Classification
Improved training strategies, dynamic weight adjustment, and better hyperparameters
"""

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import pickle
from collections import Counter
import os

# Fast.ai imports
try:
    from fastai.vision.all import *
except ImportError:
    print("Fast.ai not installed. Install with: pip install fastai")

import warnings
warnings.filterwarnings('ignore')


class DatasetPreparation:
    """Prepare dataset for both TensorFlow and Fast.ai"""

    def __init__(self, data_dir, img_size=(224, 224), test_size=0.2, val_size=0.1):
        self.data_dir = Path(data_dir)
        self.img_size = img_size
        self.test_size = test_size
        self.val_size = val_size
        self.label_encoder = LabelEncoder()

    def load_dataset(self):
        """Load and organize dataset (supports .jpg, .jpeg, .png, .bmp)"""
        images = []
        labels = []

        for blood_group_folder in self.data_dir.iterdir():
            if blood_group_folder.is_dir():
                blood_group = blood_group_folder.name
                for img_path in blood_group_folder.glob('*.*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        images.append(str(img_path.resolve()))
                        labels.append(blood_group)

        df = pd.DataFrame({'image_path': images, 'label': labels})
        print(f"Total images found: {len(df)}")
        print(f"Class distribution:\n{df['label'].value_counts()}")
        return df

    def prepare_splits(self, df):
        """Create train/val/test splits"""
        df['label_encoded'] = self.label_encoder.fit_transform(df['label'])

        train_val_df, test_df = train_test_split(
            df, test_size=self.test_size, stratify=df['label'], random_state=42
        )

        train_df, val_df = train_test_split(
            train_val_df, test_size=self.val_size/(1-self.test_size),
            stratify=train_val_df['label'], random_state=42
        )

        print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
        print(f"Classes: {self.label_encoder.classes_}")

        return train_df, val_df, test_df


class ImprovedTensorFlowCNN:
    """Enhanced TensorFlow CNN with better architecture"""

    def __init__(self, img_size=(224, 224), num_classes=4):
        self.img_size = img_size
        self.num_classes = num_classes
        self.model = None
        self.history = None

    def build_model(self):
        """Build improved CNN architecture with residual connections"""
        inputs = layers.Input(shape=(*self.img_size, 3))

        # Initial convolution
        x = layers.Conv2D(64, 7, strides=2, padding='same')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

        # Residual Block 1
        shortcut = layers.Conv2D(128, 1, strides=2, padding='same')(x)
        shortcut = layers.BatchNormalization()(shortcut)

        x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(128, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.3)(x)

        # Residual Block 2
        shortcut = layers.Conv2D(256, 1, strides=2, padding='same')(x)
        shortcut = layers.BatchNormalization()(shortcut)

        x = layers.Conv2D(256, 3, strides=2, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(256, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.3)(x)

        # Residual Block 3
        shortcut = layers.Conv2D(512, 1, strides=2, padding='same')(x)
        shortcut = layers.BatchNormalization()(shortcut)

        x = layers.Conv2D(512, 3, strides=2, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(512, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)

        # Global pooling and dense layers
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001))(x)
        x = layers.Dropout(0.4)(x)

        outputs = layers.Dense(self.num_classes, activation='softmax')(x)

        self.model = Model(inputs, outputs, name='Improved_TensorFlow_CNN')
        return self.model

    def compile_model(self, learning_rate=0.0005):
        """Compile with better optimizer for generalization"""
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

    def create_data_generators(self, train_df, val_df, batch_size=32):
        """Create enhanced data generators with stronger augmentation"""
        train_datagen = keras.preprocessing.image.ImageDataGenerator(
            rescale=1./255,
            rotation_range=30,
            width_shift_range=0.25,
            height_shift_range=0.25,
            horizontal_flip=True,
            vertical_flip=True,
            zoom_range=0.25,
            shear_range=0.2,
            brightness_range=[0.8, 1.2],
            fill_mode='nearest'
        )

        val_datagen = keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

        train_gen = train_datagen.flow_from_dataframe(
            train_df,
            x_col='image_path',
            y_col='label_encoded',
            target_size=self.img_size,
            batch_size=batch_size,
            class_mode='raw',
            shuffle=True
        )

        val_gen = val_datagen.flow_from_dataframe(
            val_df,
            x_col='image_path',
            y_col='label_encoded',
            target_size=self.img_size,
            batch_size=batch_size,
            class_mode='raw',
            shuffle=False
        )

        return train_gen, val_gen

    def train(self, train_gen, val_gen, epochs=80, callbacks_list=None):
        """Train the model"""
        self.history = self.model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=callbacks_list,
            verbose=1
        )
        return self.history

    def predict_proba(self, images):
        """Get probability predictions"""
        return self.model.predict(images, verbose=0)

    def save(self, path):
        """Save model"""
        self.model.save(path)

    def load(self, path):
        """Load model"""
        self.model = keras.models.load_model(path)


class ImprovedFastAiModel:
    """Enhanced Fast.ai Model with better training strategy"""

    def __init__(self, data_dir, img_size=224, batch_size=32):
        self.data_dir = Path(data_dir)
        self.img_size = img_size
        self.batch_size = batch_size
        self.learner = None

    def create_dataloader(self, train_df, val_df):
        """Create Fast.ai DataLoader with enhanced augmentation"""
        train_df_copy = train_df.copy()
        val_df_copy = val_df.copy()
        train_df_copy['is_valid'] = False
        val_df_copy['is_valid'] = True

        combined_df = pd.concat([train_df_copy, val_df_copy])
        combined_df['image_path'] = combined_df['image_path'].apply(lambda x: os.path.abspath(x))

        dls = ImageDataLoaders.from_df(
            combined_df,
            path='/',
            fn_col='image_path',
            label_col='label',
            valid_col='is_valid',
            item_tfms=Resize(self.img_size),
            batch_tfms=aug_transforms(
                mult=2.0,  # Increased augmentation
                do_flip=True,
                flip_vert=True,  # Added vertical flip
                max_rotate=30,
                max_zoom=1.3,
                max_lighting=0.4,
                max_warp=0.25,
                p_affine=0.8,
                p_lighting=0.8
            ),
            bs=self.batch_size
        )

        return dls

    def build_learner(self, dls, arch=resnet50):
        """Build Fast.ai learner with mixup and label smoothing"""
        self.learner = vision_learner(
            dls,
            arch,
            metrics=[accuracy, error_rate],
            pretrained=True,
            loss_func=LabelSmoothingCrossEntropy()
        )

        # Add mixup callback for better generalization
        self.learner.add_cb(MixUp())

        return self.learner

    def find_lr(self):
        """Find optimal learning rate"""
        return self.learner.lr_find()

    def train(self, epochs=40, base_lr=1e-3):
        """Enhanced training with discriminative learning rates"""
        print(f"Training Fast.ai model for {epochs} epochs...")

        # Phase 1: Train head with frozen backbone (10 epochs)
        freeze_epochs = 10
        print(f"\nPhase 1: Training head ({freeze_epochs} epochs)")
        self.learner.freeze()
        self.learner.fit_one_cycle(freeze_epochs, base_lr)

        # Phase 2: Gradually unfreeze and train (remaining epochs)
        remaining_epochs = epochs - freeze_epochs
        if remaining_epochs > 0:
            print(f"\nPhase 2: Fine-tuning all layers ({remaining_epochs} epochs)")
            self.learner.unfreeze()
            # Use discriminative learning rates
            self.learner.fit_one_cycle(
                remaining_epochs,
                lr_max=slice(base_lr/100, base_lr/10)
            )

    def predict_proba_from_paths(self, image_paths):
        """Get probability predictions from image paths"""
        predictions = []
        for img_path in image_paths:
            pred_class, pred_idx, probs = self.learner.predict(img_path)
            predictions.append(probs.numpy())
        return np.array(predictions)

    def save(self, path):
        """Save model"""
        self.learner.export(path)

    def load(self, path):
        """Load model"""
        self.learner = load_learner(path)


class DynamicEnsembleVoting:
    """Enhanced ensemble with dynamic weight adjustment"""

    def __init__(self, models, weights=None, voting='soft'):
        self.models = models
        self.weights = weights if weights else [1.0] * len(models)
        self.voting = voting

    def predict_proba(self, images_tf, image_paths_fastai):
        """Get ensemble probability predictions"""
        all_predictions = []

        # TensorFlow CNN predictions
        tf_probs = self.models[0].predict_proba(images_tf)
        all_predictions.append(tf_probs)

        # Fast.ai predictions
        fastai_probs = self.models[1].predict_proba_from_paths(image_paths_fastai)
        all_predictions.append(fastai_probs)

        if self.voting == 'soft':
            # Weighted average of probabilities
            weighted_probs = np.zeros_like(all_predictions[0])
            for pred, weight in zip(all_predictions, self.weights):
                weighted_probs += pred * weight
            weighted_probs /= sum(self.weights)
            return weighted_probs
        else:  # hard voting
            class_preds = [np.argmax(pred, axis=1) for pred in all_predictions]
            ensemble_preds = []
            for i in range(len(class_preds[0])):
                votes = [pred[i] for pred in class_preds]
                ensemble_preds.append(Counter(votes).most_common(1)[0][0])
            return np.array(ensemble_preds)

    def predict(self, images_tf, image_paths_fastai):
        """Get final class predictions"""
        if self.voting == 'soft':
            probs = self.predict_proba(images_tf, image_paths_fastai)
            return np.argmax(probs, axis=1)
        else:
            return self.predict_proba(images_tf, image_paths_fastai)

    def optimize_weights(self, images_tf, image_paths_fastai, y_true):
        """Find optimal weights based on validation performance"""
        print("\nOptimizing ensemble weights...")

        # Get individual model predictions
        tf_probs = self.models[0].predict_proba(images_tf)
        fastai_probs = self.models[1].predict_proba_from_paths(image_paths_fastai)

        # Try different weight combinations
        best_accuracy = 0
        best_weights = [0.5, 0.5]

        for w1 in np.arange(0.3, 0.8, 0.05):
            w2 = 1 - w1
            weighted_probs = tf_probs * w1 + fastai_probs * w2
            preds = np.argmax(weighted_probs, axis=1)
            acc = accuracy_score(y_true, preds)

            if acc > best_accuracy:
                best_accuracy = acc
                best_weights = [w1, w2]

        self.weights = best_weights
        print(f"Optimized weights: TensorFlow={best_weights[0]:.3f}, Fast.ai={best_weights[1]:.3f}")
        print(f"Best validation accuracy: {best_accuracy:.4f}")

        return best_weights


class EnsemblePipeline:
    """Complete ensemble training and evaluation pipeline"""

    def __init__(self, data_dir, img_size=(224, 224), save_dir='saved_models'):
        self.data_dir = data_dir
        self.img_size = img_size
        self.save_dir = save_dir
        self.data_prep = DatasetPreparation(data_dir, img_size)
        self.tf_model = None
        self.fastai_model = None
        self.ensemble = None
        self.label_encoder = None

    def prepare_data(self):
        """Prepare dataset"""
        print("Loading and preparing dataset...")
        df = self.data_prep.load_dataset()
        train_df, val_df, test_df = self.data_prep.prepare_splits(df)
        self.label_encoder = self.data_prep.label_encoder

        return train_df, val_df, test_df

    def train_tensorflow_model(self, train_df, val_df, epochs=80, batch_size=32):
        """Train improved TensorFlow CNN"""
        print("\n" + "="*60)
        print("TRAINING IMPROVED TENSORFLOW CNN MODEL")
        print("="*60)

        num_classes = len(self.label_encoder.classes_)
        self.tf_model = ImprovedTensorFlowCNN(self.img_size, num_classes)
        self.tf_model.build_model()
        self.tf_model.compile_model(learning_rate=0.0005)

        print(self.tf_model.model.summary())

        train_gen, val_gen = self.tf_model.create_data_generators(
            train_df, val_df, batch_size
        )

        # Enhanced callbacks
        callbacks_list = [
            callbacks.EarlyStopping(
                monitor='val_accuracy',
                patience=15,
                restore_best_weights=True,
                verbose=1
            ),
            callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.3,
                patience=7,
                min_lr=1e-7,
                verbose=1
            ),
            callbacks.ModelCheckpoint(
                os.path.join(self.save_dir, 'tf_best_model.h5'),
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            )
        ]

        history = self.tf_model.train(train_gen, val_gen, epochs, callbacks_list)
        return history

    def train_fastai_model(self, train_df, val_df, epochs=40, batch_size=32):
        """Train improved Fast.ai ResNet"""
        print("\n" + "="*60)
        print("TRAINING IMPROVED FAST.AI RESNET MODEL")
        print("="*60)

        self.fastai_model = ImprovedFastAiModel(self.data_dir, self.img_size[0], batch_size)

        dls = self.fastai_model.create_dataloader(train_df, val_df)
        self.fastai_model.build_learner(dls, arch=resnet50)

        print("Finding optimal learning rate...")
        lr_suggestion = self.fastai_model.find_lr()

        self.fastai_model.train(epochs=epochs, base_lr=2e-3)

        return self.fastai_model.learner

    def create_ensemble(self, val_df, weights=None, voting='soft', optimize=True):
        """Create ensemble model with optional weight optimization"""
        print("\n" + "="*60)
        print("CREATING DYNAMIC ENSEMBLE MODEL")
        print("="*60)

        self.ensemble = DynamicEnsembleVoting(
            models=[self.tf_model, self.fastai_model],
            weights=weights if weights else [0.5, 0.5],
            voting=voting
        )

        # Optimize weights on validation set
        if optimize:
            val_images_tf = []
            val_paths = val_df['image_path'].tolist()

            for img_path in val_paths:
                img = keras.preprocessing.image.load_img(img_path, target_size=self.img_size)
                img_array = keras.preprocessing.image.img_to_array(img) / 255.0
                val_images_tf.append(img_array)

            val_images_tf = np.array(val_images_tf)
            y_val = val_df['label_encoded'].values

            self.ensemble.optimize_weights(val_images_tf, val_paths, y_val)

        return self.ensemble

    def evaluate_models(self, test_df):
        """Evaluate individual models and ensemble"""
        print("\n" + "="*60)
        print("EVALUATION ON TEST SET")
        print("="*60)

        test_images_tf = []
        test_paths = test_df['image_path'].tolist()

        for img_path in test_paths:
            img = keras.preprocessing.image.load_img(img_path, target_size=self.img_size)
            img_array = keras.preprocessing.image.img_to_array(img) / 255.0
            test_images_tf.append(img_array)

        test_images_tf = np.array(test_images_tf)
        y_true = test_df['label_encoded'].values

        results = {}

        # Evaluate TensorFlow model
        print("\n--- TensorFlow CNN ---")
        tf_preds = np.argmax(self.tf_model.predict_proba(test_images_tf), axis=1)
        tf_acc = accuracy_score(y_true, tf_preds)
        results['tensorflow'] = {
            'accuracy': tf_acc,
            'predictions': tf_preds,
            'report': classification_report(y_true, tf_preds, target_names=self.label_encoder.classes_)
        }
        print(f"Accuracy: {tf_acc:.4f}")
        print(results['tensorflow']['report'])

        # Evaluate Fast.ai model
        print("\n--- Fast.ai ResNet ---")
        fastai_probs = self.fastai_model.predict_proba_from_paths(test_paths)
        fastai_preds = np.argmax(fastai_probs, axis=1)
        fastai_acc = accuracy_score(y_true, fastai_preds)
        results['fastai'] = {
            'accuracy': fastai_acc,
            'predictions': fastai_preds,
            'report': classification_report(y_true, fastai_preds, target_names=self.label_encoder.classes_)
        }
        print(f"Accuracy: {fastai_acc:.4f}")
        print(results['fastai']['report'])

        # Evaluate Ensemble
        print("\n--- Ensemble Model ---")
        ensemble_preds = self.ensemble.predict(test_images_tf, test_paths)
        ensemble_acc = accuracy_score(y_true, ensemble_preds)
        results['ensemble'] = {
            'accuracy': ensemble_acc,
            'predictions': ensemble_preds,
            'report': classification_report(y_true, ensemble_preds, target_names=self.label_encoder.classes_),
            'weights': self.ensemble.weights
        }
        print(f"Accuracy: {ensemble_acc:.4f}")
        print(f"Ensemble weights: TensorFlow={self.ensemble.weights[0]:.3f}, Fast.ai={self.ensemble.weights[1]:.3f}")
        print(results['ensemble']['report'])

        return results

    def plot_comparison(self, results):
        """Plot model comparison"""
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        models = ['tensorflow', 'fastai', 'ensemble']
        titles = ['TensorFlow CNN', 'Fast.ai ResNet', 'Ensemble']

        for idx, (model_name, title) in enumerate(zip(models, titles)):
            cm = confusion_matrix(
                results['y_true'],
                results[model_name]['predictions']
            )

            sns.heatmap(
                cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=self.label_encoder.classes_,
                yticklabels=self.label_encoder.classes_,
                ax=axes[idx]
            )
            axes[idx].set_title(f'{title}\nAccuracy: {results[model_name]["accuracy"]:.4f}')
            axes[idx].set_ylabel('True Label')
            axes[idx].set_xlabel('Predicted Label')

        plt.tight_layout()
        plt.savefig('ensemble_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()

    def save_pipeline(self):
        """Save all models and encoders to Google Drive"""
        save_dir = Path(self.save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        print(f"\nSaving models to: {save_dir}")

        # Save TensorFlow model
        tf_path = save_dir / 'tensorflow_model.h5'
        self.tf_model.save(tf_path)
        print(f"✓ TensorFlow model saved: {tf_path}")

        # Save Fast.ai model
        fastai_path = save_dir / 'fastai_model.pkl'
        self.fastai_model.save(fastai_path)
        print(f"✓ Fast.ai model saved: {fastai_path}")

        # Save label encoder
        encoder_path = save_dir / 'label_encoder.pkl'
        with open(encoder_path, 'wb') as f:
            pickle.dump(self.label_encoder, f)
        print(f"✓ Label encoder saved: {encoder_path}")

        # Save ensemble config
        config = {
            'weights': self.ensemble.weights,
            'voting': self.ensemble.voting,
            'classes': self.label_encoder.classes_.tolist(),
            'img_size': self.img_size,
            'tf_epochs': self.tf_model.history.epoch[-1] + 1 if self.tf_model.history else 0
        }
        config_path = save_dir / 'ensemble_config.json'
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=4)
        print(f"✓ Ensemble config saved: {config_path}")

        # Save training history
        if self.tf_model.history:
            history_path = save_dir / 'training_history.pkl'
            with open(history_path, 'wb') as f:
                pickle.dump(self.tf_model.history.history, f)
            print(f"✓ Training history saved: {history_path}")

        print(f"\n✅ All models successfully saved to Google Drive!")
        print(f"📁 Location: {save_dir}")


def main():
    """Main execution pipeline"""

    # Mount Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted successfully!")
        # Save to Google Drive
        SAVE_DIR = '/content/drive/MyDrive/fingerprint_models'
    except:
        print("Not running in Colab or Drive mount failed. Using local storage.")
        SAVE_DIR = 'saved_models'

    # IMPROVED CONFIGURATION
    DATA_DIR = '/content/f_dataset'  # Change to your dataset path
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 32
    TF_EPOCHS = 25       # Increased from 65
    FASTAI_EPOCHS = 80    # Increased from 28

    print("="*60)
    print("IMPROVED MULTI-MODEL ENSEMBLE PIPELINE")
    print("Enhanced TensorFlow CNN + Improved Fast.ai ResNet")
    print("Dynamic Weight Optimization")
    print("="*60)

    # Initialize pipeline
    pipeline = EnsemblePipeline(DATA_DIR, IMG_SIZE, SAVE_DIR)

    # Prepare data
    train_df, val_df, test_df = pipeline.prepare_data()

    # Train TensorFlow model
    tf_history = pipeline.train_tensorflow_model(
        train_df, val_df,
        epochs=TF_EPOCHS,
        batch_size=BATCH_SIZE
    )

    # Train Fast.ai model
    fastai_learner = pipeline.train_fastai_model(
        train_df, val_df,
        epochs=FASTAI_EPOCHS,
        batch_size=BATCH_SIZE
    )

    # Create ensemble with automatic weight optimization
    ensemble = pipeline.create_ensemble(
        val_df,
        voting='soft',
        optimize=True  # Automatically find best weights
    )

    # Evaluate all models
    results = pipeline.evaluate_models(test_df)
    results['y_true'] = test_df['label_encoded'].values

    # Plot comparison
    pipeline.plot_comparison(results)

    # Save everything
    pipeline.save_pipeline()

    print("\n" + "="*60)
    print("PIPELINE COMPLETE!")
    print("="*60)
    print(f"TensorFlow CNN Accuracy: {results['tensorflow']['accuracy']:.4f}")
    print(f"Fast.ai ResNet Accuracy: {results['fastai']['accuracy']:.4f}")
    print(f"Ensemble Accuracy: {results['ensemble']['accuracy']:.4f}")
    print(f"Optimized weights: TensorFlow={results['ensemble']['weights'][0]:.3f}, "
          f"Fast.ai={results['ensemble']['weights'][1]:.3f}")


if __name__ == "__main__":
    main()

In [ ]:
"""
Fingerprint Blood Group Classification - Visual Ensemble Inference
Automatically predicts and visualizes class probabilities for all images in a folder
"""

# ==============================
# IMPORTS
# ==============================
import tensorflow as tf
from fastai.vision.all import *
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
import pickle
from pathlib import Path
from google.colab import drive

# ==============================
# CONNECT TO GOOGLE DRIVE
# ==============================
print("🔗 Connecting to Google Drive...")
drive.mount('/content/drive')
print("✅ Drive connected!\n")

# ==============================
# CONFIGURATION
# ==============================
MODELS_DIR = '/content/drive/MyDrive/fingerprint_models'      # trained models folder
DATA_DIR = '/content/drive/MyDrive/fingerprint_samples'       # folder with test images
IMG_SIZE = (224, 224)

# ==============================
# FUNCTIONS
# ==============================
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    """Load and resize an image"""
    img = Image.open(image_path).convert('RGB')
    img = img.resize(target_size)
    img_array = np.array(img)
    return img, img_array

def predict_with_tensorflow_model(model, img_array):
    """Predict using TensorFlow model"""
    img_tf = img_array.astype('float32') / 255.0
    img_tf = np.expand_dims(img_tf, axis=0)
    tf_pred = model.predict(img_tf, verbose=0)
    return np.argmax(tf_pred[0]), tf_pred[0]

def predict_with_fastai_model(learn, img_array):
    """Predict using FastAI model"""
    fastai_img = PILImage.create(Image.fromarray(img_array))
    pred, pred_idx, probs = learn.predict(fastai_img)
    return pred_idx, probs

def ensemble_predictions(tf_probs, fastai_probs, weights=(0.3, 0.7)):
    """Weighted ensemble prediction"""
    combined_probs = (weights[0] * tf_probs + weights[1] * fastai_probs.numpy())
    final_class = np.argmax(combined_probs)
    final_confidence = combined_probs[final_class]
    return final_class, combined_probs

def visualize_prediction(img, ensemble_probs, label_encoder, final_class, confidence, image_name):
    """Display image and class probability bar chart"""
    plt.figure(figsize=(10, 5))

    # Show fingerprint image
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"{image_name}", fontsize=12, fontweight='bold', color='#2F4F4F')
    plt.axis('off')

    # Bar chart of class probabilities
    class_names = label_encoder.classes_
    colors = ['#2E8B57' if i == final_class else '#4682B4' for i in range(len(ensemble_probs))]
    plt.subplot(1, 2, 2)
    bars = plt.bar(range(len(ensemble_probs)), ensemble_probs, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)

    plt.title('Class Probabilities', fontsize=14, fontweight='bold', color='#2F4F4F')
    plt.xlabel('Classes', fontsize=12, fontweight='bold')
    plt.ylabel('Confidence', fontsize=12, fontweight='bold')
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right', fontsize=9)
    plt.ylim(0, 1.1)
    plt.grid(axis='y', alpha=0.3, linestyle='--')

    # Add text labels
    for bar, prob in zip(bars, ensemble_probs):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02, f'{prob:.2f}', ha='center', va='bottom', fontsize=9)

    plt.suptitle(f"Predicted: {class_names[final_class]} ({confidence*100:.2f}%)",
                 fontsize=14, fontweight='bold', color='#2E8B57', y=1.03)
    plt.tight_layout()
    plt.show()

# ==============================
# MAIN
# ==============================
def main():
    print("🔄 Loading models and encoders...")

    # Load TensorFlow and FastAI models
    tf_model_path = os.path.join(MODELS_DIR, 'tensorflow_model.h5')
    fastai_model_path = os.path.join(MODELS_DIR, 'fastai_model.pkl')
    label_path = os.path.join(MODELS_DIR, 'label_encoder.pkl')

    if not all(os.path.exists(p) for p in [tf_model_path, fastai_model_path, label_path]):
        print("❌ One or more model files missing in:", MODELS_DIR)
        return

    tf_model = tf.keras.models.load_model(tf_model_path)
    learn = load_learner(fastai_model_path)

    with open(label_path, 'rb') as f:
        label_encoder = pickle.load(f)

    print("✅ Models loaded successfully!\n")

    # ✅ Get all valid image files (case-insensitive)
    valid_extensions = ('.bmp', '.jpg', '.jpeg', '.png')
    image_paths = [p for p in Path(DATA_DIR).glob('*') if p.suffix.lower() in valid_extensions]

    if not image_paths:
        print("❌ No images found in the folder:", DATA_DIR)
        return

    print(f"📸 Found {len(image_paths)} images in '{DATA_DIR}'\n")

    # Predict and visualize for each image
    for img_path in image_paths:
        print(f"🔹 Processing {img_path.name}...")
        img, img_array = load_and_preprocess_image(img_path)
        tf_class, tf_probs = predict_with_tensorflow_model(tf_model, img_array)
        fastai_class, fastai_probs = predict_with_fastai_model(learn, img_array)
        ensemble_class, ensemble_probs = ensemble_predictions(tf_probs, fastai_probs, weights=(0.3, 0.7))
        confidence = ensemble_probs[ensemble_class]
        visualize_prediction(img, ensemble_probs, label_encoder, ensemble_class, confidence, img_path.name)

# ==============================
# RUN
# ==============================
if __name__ == "__main__":
    main()


Output hidden; open in https://colab.research.google.com to view.